# Machine Learning to Analyze NOvA Near Detector Events
Welcome back! This is the last notebook of the set, and it ties the others together.

**What you've already done:**
- In the **Near Detector Analysis** notebook, you took a track length cutoff you'd agreed on from the Far Detector event images, counted the events above it, and measured a Near Detector ratio.

- In the **Pulsar** notebook, you taught a Decision Tree to separate real pulsars from radio junk. You hand-drew a cut, watched sklearn find nearly the same one, broke a model on purpose to see overfitting, and argued about efficiency vs. purity.

**Now those two things meet.** With analysis pointed at neutrinos.

To "Run" a cell, put your cursor in it and press "Shift and Enter" at the same time.

In [ ]:
# Let's run our imports, starting here with the usual data anlaysis tools
import pandas as pd   # organizing and formatting data
import numpy as np    # math and statistics
import matplotlib as mpl
import matplotlib.pyplot as plt

# ...and the machine learning toolkit, same as the pulsar notebook.
from sklearn.tree import DecisionTreeClassifier, plot_tree #The model we'll train
from sklearn.model_selection import train_test_split #For splitting our data honestly
from sklearn.metrics import confusion_matrix #For grading our model

print("Packages Imported!")

# Part One: Review the Near Detector Data Set
Load the Near Detector data -- the same file you used in the last notebook.

Then let's take another look at what it contains

In [ ]:
# Same near detector file as before.

nearData = pd.read_csv('NOvA-ND-Events-Data.csv')
nearData.head()

In [ ]:
# As a first look, make a call to "describe"
nearData.describe()

# Part Two: Look at the Data...and What's Missing...
Listing each column name here...pause to ask **what does each column represent?**

*   Event Number
*   Longest Track Length (cm)
*   2nd Longest Track Length (cm)
*   3rd Longest Longest Track Length (cm)

Be sure to pause here for a moment to describe them.

Also...

What Column is Missing from the Near Detector Data that you did have in the pulsar data set?
___
___

**Some Discussion of the Columns**

Every column is a **measurement**: how long was the longest track, the 2nd longest, the 3rd longest.

Not one column says **what the event was.** No "CC" column. No "NC" column.  No "Result" column.

Compare that to the pulsar notebook. There, every candidate had a `target` column -- 1 for pulsar, 0 for junk -- because **astronomers pointed telescopes at those candidates and checked.** We can go back to the pulsar canndidate and visually look at it, confirming that it is indeed a pulsar.

Neutrino interactions are not as straight forward.  The interaction happens, but we never actually see the neutrino.  We suspect...perhaps have a probability...but our neutrino data does not have a definite "it is _______" result tied to it.

Our supervised, decision tree machine learning model does need a "target" column that states what type of event the neutrino (or pulsar) line in the data is.  Continue on to see how we can accomplish this.

# Part Three: Where the Answer Key Comes From
Here's the source of our NOvA "Answer Key"

> **The Training Data is Based on a Simulation**

We know the Standard Model. We know how often a neutrino makes a muon, how far a muon of a given energy travels, how a hadronic spray spreads. So you run that physics forward in a computer, generate millions of fake events, and **because you built them, you know exactly what each one is.**

Then you train on the simulation and point the trained model at reality.

Now, NOvA's real classifier is trained on Monte Carlo, simulated data from GENIE and GEANT4.

We're going to do the same thing here, albiet with a much simpler simulation, though.

The training, simulation data used here comes from a Python program that was developed with a much smaller look at the physics of NOvA neutrino events.

**The handful of physics ideas our sketch is built on:**
- **CC numu:** the neutrino turns into a **muon**. Muons are minimum-ionizing -- they lose about **2 MeV for every cm** of NOvA scintillator, so a muon's range in cm is roughly its energy in MeV over 2. A 1 GeV muon travels about **5 meters**. That one line is why the CC images look the way they do.
- **CC also makes a proton:** the interaction is `nu_mu + n -> mu- + p`, so most CC events carry a short recoil-proton track next to the long muon.
- **NC:** the neutrino stays a neutrino and leaves. No muon. Just a stubby hadronic spray from the struck nucleus.

Next, let's load in our training data set...

In [ ]:
# The SIMULATED training data. A different file from the near detector data!

trainData = pd.read_csv('nova_training_data.csv')
# Note the name of the object that stores the data file:  trainData
trainData.head()

In [ ]:
# As is a typical first step, make a "describe" call
trainData.des____

# Count the Event Types in the Training Data

Now that we have Event Types, and we use a 1 and 0 to represent...


*   1 = Charged Current Muon Neutrino Event
*   0 = Neutral Current Neutrino Event

We can take a look and see how many of each event type are present.

Below is a copy of a similar cell from our "Pulsar" analysis.  Update it for our Neutrino trainingData

In [ ]:
# How many total events & how many of each kind of candidate do we have?

# Be sure to update the data frame name to "trainData"
totalNumEvents = len(data['target']) # len returns how many lines are in a column

# Instead of Pulsars and Junk, maybe numuCCEvents and neutralNCEvents or something similar for a variable name
numPulsars = np.sum(data['t_____']) # remember, a value of 1 is a pulsar
numJunk = ***some math here using above variables to determine number of junk events***

#Update the references here too for printing out the results
print("Number of Events", totalNumEvents)
print("Number of Pulsars", numPul____)
print("Number of /'Junk/' events", numJu_____)

# What Do You Observe?

*   How do the number of CCNuMu events compare to the number of NC neutral events?


*   Think back to the Far Detector images you looked at...are the event type numbers seen here in line with the Far Detector?

___
___

# Part Four:  Deeper Look at the Training Data
Following the discussion points above, you should find that the training data set is about **half CC and half NC.**

But wait -- the Far Detector images you analyzed were **6 CC and 40 NC.** Only about 13% CC. Shouldn't the training data match reality?

**No. And getting this wrong could quietly ruin the whole measurement.**

Think about what you're trying to find out. You want to know **how many CC events the Near Detector has.**

Now suppose you trained on a set that was 13% CC. You'd be teaching the model *"CC events are rare -- when in doubt, guess NC."*

But they wouldn't have. **You'd have assumed your answer and gotten it back.** You'd have baked the Far Detector's oscillation into your Near Detector measurement, possibly interpreting it as a result.

On the other hand, the pulsar data you looked at was ~9% actual pulsars and 91% junk, and we did not train on a 50/50 type of split.  The ratio there was kept.  However, that training data was *the data* where as our NOvA near detector training set is simulated, so any split we choose would not be the correct one.

So we go 50/50. The training data then teaches the model only what CC and NC **look like** -- the shape of the thing -- and says nothing at all about **how many** to expect. The counting is left to the data.

Pause for a moment here...

- In your own words:  Explain why we should use a 50/50 CC vs NC in the NOvA training data set
- Describe the difference between using the 9/91 split for training the pulsar data vs the 50/50 for NOvA

# Part Five: Does the Simulation Look Like Reality?
Before trusting a simulation you check it against the real world.  If your simulation can't reproduce the data, nothing downstream of it means anything.

Run the cell. It overlays our simulated events on the real Near Detector events.
- Do the shapes match?
- Where do they disagree?

In [ ]:
plt.figure(figsize=(9,5))
# Update the max range value in the histogram calls here.  Refer to the "describe" output & column of interest to help with this
plt.hist(nearData['Longest Track Length (cm)'], bins=40, range=[0,__maxRangeValue__],
         histtype='step', linewidth=2, label='REAL near detector data', density=True) # Remember Density "normalizes" the output
plt.hist(trainData['Longest Track Length (cm)'], bins=40, range=[0,__maxRangeValue__],
         histtype='step', linewidth=2, label='our simulation (50/50 mix)', density=True)
plt.xlabel("Longest Track Length (cm)")
plt.ylabel("fraction of events")
plt.title("Simulation vs. Reality")
plt.legend()
plt.grid(False);

# These won't match perfectly, and they SHOULDN'T.
# Our simulation is 50/50 CC/NC. The real near detector has whatever mix nature gave it.
# That difference is not a bug. It's the thing we're about to measure.

# Part Six: What CC and NC Look Like, Separately
Now the payoff of having labels: split the simulation by truth.

**This is the picture you can never draw for the real data.**

In [ ]:
simCC = trainData[trainData['Event Type'] == __valueForChargedCurrentEventType__]
simNC = trainData[trainData['Event Type'] == __valueForNeutralCurrentEventType__]

plt.figure(figsize=(9,5))
plt.hist(simNC['Longest Track Length (cm)'], bins=40, range=[0,__maxRangeValue__], alpha=0.6, label='NC (no muon)')
plt.hist(simCC['Longest Track Length (cm)'], bins=40, range=[0,__maxRangeValue__], alpha=0.6, label='CC numu (has a muon!)')
plt.xlabel("Longest Track Length (cm)")
plt.ylabel("number of events")
plt.title("Update this title to something better")
plt.legend()
plt.grid(False);

# Part Seven: Grade Your Own Cutoff
Two populations -- and they **overlap.** There is no track length that cleanly separates them. Some NC events fake a long track. Some CC muons stop early.

Remember your `trackLengthCutoff` from the last notebook? Now that we have truth labels, we can finally grade it. Same two numbers as the pulsar notebook:

- **Efficiency** = of all the real CC events, what fraction did your cutoff catch? *(Did you miss any?)*
- **Purity** = of everything your cutoff called CC, what fraction really was CC? *(Did you cry wolf?)*

Put your number in and run it.

In [ ]:
# The cutoff your group agreed on from the Far Detector event images
trackLengthCutoff = ______

iSayItsCC = trainData['Longest Tra______'] > trackLen______ # Store the events our cutoff value would classify as CC Muon Events
itReallyIsCC = trainData['Event Type'] == 1 # Store the events the training data has identified as CC Muon Events

truePositives  = np.sum(iSayItsCC & itReallyIsCC)     # caught a real CC event
falsePositives = np.sum(iSayItsCC & ~itReallyIsCC)    # an NC event fooled us
falseNegatives = np.sum(~iSayItsCC & itReallyIsCC)    # a CC event we threw away

handEfficiency = truePositives / (truePositives + falseNegatives)
handPurity = truePositives / (truePositives + falsePositives)

print("Your cutoff:", trackLengthCutoff, "cm")
print("   CC events caught:  ", truePositives)
print("   NC events mistaken:", falsePositives)
print("   CC events missed:  ", falseNegatives)
print()
print("   Efficiency:", round(handEfficiency, 3))
print("   Purity:    ", round(handPurity, 3))

# Try 300. Try 500. Can you get efficiency AND purity both above 0.95?
# Really try. Failing is the point.

# Cutoff Value Reflection
Make some notes about the efficiency and the purity as you play with the track length cutoff value.

*   Do we want "Purity" or "Efficiency" as the higher priority here?  How do you know?  Keep in mind what we are looking for (our ratio of CC to All events)
*   List a few results of cutoff value, efficiency, and purity
*   How'd we do with our cutoff value based on image analysis alone earlier?  Briefly discuss and reflect.

# Part Eight: More Thoughts on the Cuttoff Value
Something worth noticing about how you chose your cutoff from our image analysis.

You looked at **6 CC images and 40 NC images.** So you saw the NC population well and the CC population barely at all. Faced with that, the natural move is to put your cut **just above the longest NC track you saw** -- safely clear of the junk.

That's a **purity-first** choice. It's reasonable! It's also why your efficiency in the cell above is probably not great: you're throwing away real CC events to keep the sample clean, and you were probably biased by the images that were available to analyze.

**Now the honest part about those 6 CC images.**

One of them (CC image 2) has a much shorter track than the rest -- about **265 cm**, where the others run 570-790 cm. **The track from that event really is that short.**

Is it an outlier that should be thrown out?

- A 265 cm muon has energy about 265 x 2 = **530 MeV**. Is that a broken event, or just a low-energy neutrino? *(The beam peaks near 2 GeV but has a real low-energy tail.)*
- Our simulation says roughly **11%** of CC events have tracks under 300 cm. In a sample of 6, how many would you expect? *(About 0.7. The chance of getting exactly one is about 37%.)*
- **So what happens if you delete it?** Your measured efficiency jumps from 5/6 to 6/6. You'd conclude your cutoff is perfect, apply no correction, and undercount CC events in the Near Detector.

That last point is the one to sit with. **CC image 2 is the only one of the six that tells you your efficiency isn't 1.0.** That makes it the most informative event you have, not the most disposable. Deleting the inconvenient data point is the one choice guaranteed to bias your answer -- and it biases it toward hiding the oscillation.

**Take a Reflection Moment**
- Take a moment here to offer some thoughts and reflection on this cell.  Particularly return to our cutoff discussion following the Far Detector image analysis.  Does the above change anything about the result of that discussion?
- When *is* it legitimate to remove a data point? What would you need to know first?

# Part Nine: Machine Learning Time - Let the Computer Pick the Cuttoff Value
Exactly as in the pulsar notebook: train a Decision Tree, hobbled on purpose. `max_depth=1` means **one question**, on one variable. It picks which variable and where to cut. That's all.

We're asking the computer to do precisely what you did by hand earlier based on your image analysis.

In [ ]:
features = ['Longest Track Length (cm)',
            '2nd Longest Track Length (cm)',
            '3rd Longest Track Length (cm)']

X = trainData[features]        # the measurements
y = trainData['Event Type']    # the answer key

stump = DecisionTreeClassifier(max_depth=1, random_state=42)  # ONE question only!
stump.fit(X, y)

print("The computer chose to cut on:", features[stump.tree_.feature[0]])
print("It put the cut at:", round(stump.tree_.threshold[0], 1), "cm")
print()
print("Your group's cutoff was:", trackLengthCutoff, "cm")

# Note the output from the Decision Tree Run
- Which variable did it choose?
- Where did it put the cut? How close is that to your group's number?

# Part Ten: Sit With That For a Second
A machine learning model is not magic and it is not a brain.

**A decision tree is a machine that finds cuts.** It tried a pile of cut values, scored each, kept the best. You could do it with a ruler and a lot of patience -- you basically did, with 46 images.

The tree probably picked a **lower** cut than your group did. That's not the tree being clever. It's the tree not having been shown 40 NC images and 6 CC images, so it never developed your (very sensible) fear of contamination.

Neither of you is "right" yet.

# Part Eleven: Training Data vs. Testing Data
Same rule as the pulsar notebook:

> **You cannot use the same events to build your cut and to report how well it works.**

Tune on some events and grade yourself on those same events, and you haven't measured your method -- you've measured your own tuning.

- `stratify=y` keeps the same CC/NC balance in both halves.
- `random_state=42` makes the split reproducible so the whole room matches. **Change it to 1, 2, 3** and watch the scores wobble. That wobble is real: it's the statistical uncertainty on your measurement, and it's what error bars are for.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,       # lock away 30% for the exam
    random_state=42,     # try 1, 2, 3...
    stratify=y           # keep the CC/NC balance in both halves
)

print("Training events:", len(X_train))
print("Testing events: ", len(X_test))
print()
print("CC fraction in training set:", round(y_train.mean(), 3))
print("CC fraction in testing set: ", round(y_test.mean(), 3))

# Part Twelve: Break It On Purpose
Take the leash off. A tree with no depth limit can keep asking questions until every training event is sorted perfectly.

**Write down your prediction** for the two numbers below before running it.

In [ ]:
greedyTree = DecisionTreeClassifier(random_state=42)   # no limit!
greedyTree.fit(X_train, y_train)

print("Accuracy on the TRAINING events:", round(greedyTree.score(X_train, y_train), 4))
print("Accuracy on the TESTING events: ", round(greedyTree.score(X_test, y_test), 4))
print()
print("How many questions deep did it go?", greedyTree.get_depth())

# It Memorized. It Didn't Learn.
A perfect score on training, a worse score on the exam. The tree didn't learn what a muon is -- it memorized the events we showed it, right down to the noise.

That gap is **overfitting**, the single most common way to fool yourself with machine learning. Same thing you saw with the pulsars.

Put the leash back on. Three questions.
- **Predict again:** training accuracy drops. What does testing accuracy do?

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

print("Accuracy on the TRAINING events:", round(tree.score(X_train, y_train), 4))
print("Accuracy on the TESTING events: ", round(tree.score(X_test, y_test), 4))

**The simpler model fits the training data worse and does better on the exam.** Not a typo. A model that's too flexible chases noise. Restraint is a feature.

# Part Thirteen: Read the Tree Out Loud
This is why we use a Decision Tree and not something fancier. **You can just look at it.**

Read the top box out loud as an English sentence. It's a cut on a track length -- the same kind of statement you wrote in your pulsar notebook previously.
- Does the tree's first question match the physics you'd expect?
- Follow one path top to bottom. Can you narrate the whole decision?

In [ ]:
plt.figure(figsize=(18,9))
plot_tree(tree,
          feature_names=['longest', '2nd longest', '3rd longest'],
          class_names=['NC', 'CC numu'],
          filled=True, rounded=True, fontsize=9)
plt.show()

# 'value' shows [how many NC, how many CC] landed in each box.
# 'gini' measures how mixed-up a box is. 0.0 means perfectly sorted.

# Part Fourteen: How Good Is It, Really?
Accuracy hides the story. Get the same two numbers you used on your hand cutoff -- but now on the locked-away testing events.

**Compare to your hand cutoff's scores from Part Seven. Did the machine beat you?**

In [ ]:
predictions = tree.predict(X_test)
cm = confusion_matrix(y_test, predictions)
trueNeg, falsePos, falseNeg, truePos = cm.ravel()

print("Correctly called NC:  ", trueNeg)
print("NC mistaken for CC:   ", falsePos, "  <-- contamination in our signal")
print("CC events MISSED:     ", falseNeg, "  <-- real muon neutrinos, thrown away")
print("Correctly called CC:  ", truePos)
print()
modelEfficiency = truePos / (truePos + falseNeg)
modelPurity = truePos / (truePos + falsePos)
print("Model Efficiency:", round(modelEfficiency, 3))
print("Model Purity:    ", round(modelPurity, 3))

Notice those two mistakes are **not** the same mistake.
- A false positive puts an NC event in your signal sample. Contamination.
- A false negative throws away a real muon neutrino. Gone.

Which should you work harder to avoid? **That's not a math question.** No equation answers it. A human has to decide what the measurement is for. Machine learning doesn't get you out of that call -- it just makes the call explicit.

# Part Fifteen: Which Measurement Did the Work?
The tree had **three** measurements to choose from. Which did it lean on?

Here's the thing to have in mind before you run it. You know from the physics that a CC event is `nu_mu + n -> mu- + p` -- a muon **and** a recoil proton. So most CC events in the training data carry a short second track from that proton. You handed the tree that second track on the sensible theory that the muon-plus-proton signature would help it spot CC events.

**Did it? Run it and find out.**

In [ ]:
importance = pd.DataFrame({
    'measurement': ['longest', '2nd longest', '3rd longest'],
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.to_string(index=False))

# Well, That's Humbling
The longest track does essentially **all** the work. The 2nd and 3rd contribute almost nothing -- even though the proton is right there in nearly every CC event.

That's worth slowing down for, because it's not what you'd guess. The muon-plus-proton signature is real, and the tree still barely used the proton. Why?

- The muon is a giveaway because **only CC events have one.** A long minimum-ionizing track is unique to the signal.
- The proton is *not* a giveaway, because **NC events also have short tracks** -- their hadronic spray throws stubs of about the same length. A 100 cm second track is equally at home in a CC event (a proton) or an NC event (a pion). It doesn't tell the two apart.

So the second track is present, physical, and correct -- but also not necesssary for determining the classification of a CC muon event vs a NC event.

# Part Sixteen: Remember the Goal -- Neutrino Oscillation
Now the real thing.

The model has never seen the Near Detector data. It learned entirely from simulation. Point it at reality and count.

In [ ]:
# Apply the trained model to the REAL near detector data.
# The model has never seen a single one of these events.
nearPredictions = tree.predict(nearData[features])

nearCCMuonNeutrinos = np.sum(nearPredictions == **valueForCCMuonEvents**)
nearNCAllNeutrinos = np.sum(nearPredictions == **valueForNCEvents**)

print("Near Detector Charged Current Muon Event Count:", nearCCMuonNeutrinos)
print("Near Detector Neutral Current Event Count:     ", nearNCAllNeutrinos)
print("Total events:                                  ", len(nearData))

In [ ]:
# The ratio, same definition you used in the last notebook
nearEventRatioML = ***Math Using Variables to Calculate the Near Detector Ratio***
print("Near Detector Event Ratio (Decision Tree):", nearEventRatioML)

In [ ]:
# The Far Detector numbers from the event images you analyzed earlier (6 CC, 40 NC)
farCCMuonNeutrinos = ______   # how many far detector images were CC numu?
farNCAllNeutrinos = ______    # how many were NC?
farEventRatio = ***Math Using Variables to Calculate the Far Detector Ratio***

print("Far Detector Event Ratio: ", farEventRatio)
print("Near Detector Event Ratio:", nearEventRatioML)

In [ ]:
# A Percent Difference Calculation Could be Used
# to Help Decide if the 2 Values are Different
# Try this out using the definition...
#  ( |valueA - valueB| / (0.5(valueA + valueB ))) X 100

eventPercentDiff = (np.abs(nearEventRatioML - _________
print("Event Percent Difference: ", eventPercentDiff)

# What Did You Just Measure?
Compare those two ratios.

- The Near Detector sits about 1 km from the beam source. The Far Detector is **810 km** away. Same beam. Both counting the same kind of thing.
- If muon neutrinos just traveled 810 km and arrived, what should the two ratios be?
- They aren't. So **where did the muon neutrinos go?**

Also compare to the ratio you got by hand in the last notebook. Two methods, same data.
- Did they agree?
- If they disagree, which do you believe, and why?
- **If your conclusion survives both methods, that's a far stronger result than either alone.**

# Part Seventeen:
Your group cut around 400-450 cm. The tree cut lower. **You disagree**, and your raw CC counts differ a lot.

So who's right?

Run this. It sweeps the cutoff and shows the raw count **and** the efficiency-corrected count at each one.

> corrected CC count = (raw count x purity) / efficiency

Some real CC events get missed (efficiency < 1) so the raw count is too low. Some NC events sneak in (purity < 1) so it's too high. The simulation tells you the size of both.

In [ ]:
print(" cut |  eff  | purity | raw CC | CORRECTED CC")
for c in [250, 300, 350, 400, 450, 500]:
    a = trainData['Longest Track Length (cm)'] > c
    b = trainData['Event Type'] == 1
    tp = np.sum(a & b); fp = np.sum(a & ~b); fn = np.sum(~a & b)
    eff = tp/(tp+fn); pur = tp/(tp+fp)
    raw = np.sum(nearData['Longest Track Length (cm)'] > c)
    corrected = raw * pur / eff
    print(f" {c:4d} | {eff:.3f} | {pur:.3f} | {raw:5d}  |   {corrected:7.0f}")

# Look At Those Last Two Columns
The **raw** counts swing wildly -- close to a factor of two across the sweep. If two groups picked different cutoffs, they got very different numbers, and each probably assumed the other blew it.

The **corrected** counts barely move.

That's not luck. Once you correct for what your cut misses and what sneaks in, **it stops mattering where you put the cut.** Every group in this room, with every cutoff anyone argued for, is measuring the same underlying number.

This is why physicists care so much about efficiency and purity. It isn't bookkeeping. It's what turns an arbitrary human choice into a measurement someone else can reproduce.

- Did your cutoff and the tree's cutoff give the same *corrected* answer?
- Which cutoff would you rather use anyway? *(Think: which needs the smallest correction?)*
- **The trap:** all of this leans on efficiency and purity coming from the simulation. If the simulation is wrong, every corrected number is wrong *together* -- and they'd still agree with each other. **Would you be able to tell?**

# Part Eighteen - Final Thoughts:
Following your experience, respond to the following, concluding questions.

*   Based on your analysis, do you see evidence of neutrino oscillation?  Explain how you know and include why or why not you can definitively say oscillation occurred.
*   You used 2 methods, both for Neutrinos and for the Pulsar data, a "by hand" cut and also using a machine learning decision tree.
> * Where did they agree and where did they disagree?
> * Which method would you "trust" more (and is "trusting" the right way to think about it?)
* Now having worked through 2, Decision Tree Machine Learning models, what are some takeaways you have about AI and Machine Learning overall from this experience?
* What are some takeaways from your experience that would apply to your own classroom and teaching?



---
# Credits
This notebook is the machine learning companion to the NOvA Near Detector analysis activity by QuarkNet Neutrino Fellow **Mike Plucinski**, and follows the pattern of the [QuarkNet](https://quarknet.org/) coding activities by [Adam LaMee](https://adamlamee.github.io/) and the CMS muon mass activity.

Far Detector event images: [CC numu events](https://web.quarknet.org/mc/nova/fardet12aug2021/numuEvents.html) and [NC events](https://web.quarknet.org/mc/nova/fardet12aug2021/ncEvents.html), NOvA Masterclass materials. Track length measurements by workshop participants, Minneapolis 2026.

**The training data used here is SIMULATED.** It was generated for teaching purposes by a simplified model: muon range from dE/dx, hadronic prongs and beam spectrum tuned to match the near detector data, plus a punch-through pion component motivated by the four long-track NC images. **It is not official NOvA simulation** and must not be used for any physics result. The generator script is included so you can read exactly what it assumes and change it.

More about NOvA at [novaexperiment.fnal.gov](https://novaexperiment.fnal.gov/). Machine learning tools from [scikit-learn](https://scikit-learn.org/). More activities and license info at [CODINGinK12.org](http://www.codingink12.org).